# Phase B Step 6 Retry: Fix Failed Segments

Retries mix segments (06b) that failed during the main Step 6 run (typically CUDA OOM).
Also recomputes residuals (06c) for those segments.

**How it works:**
1. Lists all expected segment IDs from transition PKLs on HF
2. Lists all `stems/mix_segments/{tran_id}/` dirs actually on HF
3. Finds the diff (missing segments)
4. Re-runs Demucs on just the missing segments with `torch.cuda.empty_cache()` after each
5. Recomputes residuals for those segments
6. Uploads to HF

**Runtime:** GPU P100. Typically fast — only processes the handful of OOM'd segments.

**No manifest split needed** — processes ALL mixes from both parts. Run once after both Step 6 notebooks finish.

In [ ]:
!pip install -q --force-reinstall --no-deps demucs
!pip install -q huggingface_hub soundfile librosa cvxpy torchaudio ecos clarabel

import subprocess, sys, os
# Enable PyTorch memory fragmentation mitigation (fixes GPU OOM on large segments)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/Uday-461/ai-dj-v4.git', '/kaggle/working/ai-dj/v4'],
    capture_output=True, text=True
)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print('Clone failed:', result.stderr)
else:
    print('Repo ready at /kaggle/working/ai-dj/v4')

!pip install -q -e /kaggle/working/ai-dj/v4
sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PYTORCH_CUDA_ALLOC_CONF: {os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "not set")}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os

HF_TOKEN    = "hf_..."                   # <-- paste your HuggingFace token here
HF_REPO     = "Uday-4/djmix-v3"
DATA_ROOT   = "/kaggle/working/djmix"
MANIFEST    = "both"                     # "part1", "part2", or "both" to retry only specific mixes

# -------------------------------------------------------
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["AIDJ_DATA_ROOT"] = DATA_ROOT

if HF_TOKEN.startswith("hf_..."):
    raise ValueError("Edit Cell 2: set HF_TOKEN to your real HuggingFace token")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    api.whoami()
    print(f'HF auth OK — repo: {HF_REPO}')
except Exception as e:
    raise RuntimeError(f'HF auth failed: {e}')

In [ ]:
import json
import pickle
import shutil
import tempfile
from pathlib import Path
from collections import defaultdict
from huggingface_hub import hf_hub_download, list_repo_files

DATA_ROOT_PATH = Path(DATA_ROOT)

# --- Load manifest to get target mix IDs ---
manifest_part1_path = "/kaggle/working/ai-dj/v4/data/manifest_100mix_part1.json"
manifest_part2_path = "/kaggle/working/ai-dj/v4/data/manifest_100mix_part2.json"

with open(manifest_part1_path) as f:
    manifest_part1_mixes = {m["id"] for m in json.load(f)["mixes"]}
with open(manifest_part2_path) as f:
    manifest_part2_mixes = {m["id"] for m in json.load(f)["mixes"]}

if MANIFEST == "part1":
    target_mixes = manifest_part1_mixes
    print(f'Targeting manifest part1: {len(target_mixes)} mixes')
elif MANIFEST == "part2":
    target_mixes = manifest_part2_mixes
    print(f'Targeting manifest part2: {len(target_mixes)} mixes')
else:  # "both"
    target_mixes = manifest_part1_mixes | manifest_part2_mixes
    print(f'Targeting both manifests: {len(target_mixes)} mixes')

# --- List everything on HF ---
print('Listing HF files...')
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))

# Segment stem dirs on HF: stems/mix_segments/{tran_id}/{stem}.ogg
# A segment is "complete" if it has all 4 stems
seg_stem_files = [f for f in all_hf_files if f.startswith("stems/mix_segments/") and f.endswith(".ogg")]
seg_stems_by_tran = defaultdict(set)
for f in seg_stem_files:
    parts = f.split("/")
    tran_id = parts[2]
    stem_name = parts[3].replace(".ogg", "")
    seg_stems_by_tran[tran_id].add(stem_name)

EXPECTED_STEMS = {"drums", "bass", "vocals", "other"}
complete_segments = {tid for tid, stems in seg_stems_by_tran.items() if stems >= EXPECTED_STEMS}

# Residual files on HF: results/residuals/{mix_id}/{tran_id}.npz
residual_files = [f for f in all_hf_files if f.startswith("results/residuals/") and f.endswith(".npz")]
residual_tran_ids = set()
for f in residual_files:
    parts = f.split("/")
    tran_id = parts[2] + "/" + parts[3].replace(".npz", "")  # mix_id/tran_id
    residual_tran_ids.add(parts[3].replace(".npz", ""))

# Transition PKLs on HF
transition_pkls = [f for f in all_hf_files if f.startswith("results/transitions/") and f.endswith(".pkl")]

# Load both manifests to get full mix info
manifest_path_1 = "/kaggle/working/ai-dj/v4/data/manifest_100mix_part1.json"
manifest_path_2 = "/kaggle/working/ai-dj/v4/data/manifest_100mix_part2.json"
manifest_mixes = []
for mp in [manifest_path_1, manifest_path_2]:
    with open(mp) as f:
        manifest_mixes.extend(json.load(f)["mixes"])

# Download transition PKLs and find expected segment IDs
print('Downloading transition PKLs...')
trans_dir = DATA_ROOT_PATH / "results" / "transitions"
trans_dir.mkdir(parents=True, exist_ok=True)

expected_segments = {}  # tran_id -> {mix_id, tran dict}
mixes_with_transitions = {}  # mix_id -> [transitions]

for m in manifest_mixes:
    mix_id = m["id"]
    
    # Filter by manifest
    if mix_id not in target_mixes:
        continue
    
    pkl_path = trans_dir / f"{mix_id}.pkl"
    if not pkl_path.exists():
        try:
            hf_hub_download(
                repo_id=HF_REPO, filename=f"results/transitions/{mix_id}.pkl",
                repo_type="dataset", token=HF_TOKEN,
                local_dir=DATA_ROOT,
            )
        except Exception:
            continue
    if not pkl_path.exists():
        continue
    with open(pkl_path, 'rb') as f:
        transitions = pickle.load(f)
    if not transitions:
        continue
    mixes_with_transitions[mix_id] = transitions
    for t in transitions:
        tran_id = t["tran_id"]
        # Skip segments that would be too short (same logic as step_06b)
        mix_cue_in = t.get("mix_cue_in_time_next")
        mix_cue_out = t.get("mix_cue_out_time_prev")
        if mix_cue_in is None or mix_cue_out is None:
            continue
        from aidj import config
        seg_len = int((max(mix_cue_in, mix_cue_out) - min(mix_cue_in, mix_cue_out)) * config.SR)
        if seg_len < config.SR:
            continue
        expected_segments[tran_id] = {"mix_id": mix_id, "tran": t}

# Find missing segments
missing_segments = {tid: info for tid, info in expected_segments.items()
                    if tid not in complete_segments}

# Group by mix
missing_by_mix = defaultdict(list)
for tran_id, info in missing_segments.items():
    missing_by_mix[info["mix_id"]].append(info["tran"])

# Also find missing residuals for segments that DO exist
missing_residuals_by_mix = defaultdict(list)
for tran_id, info in expected_segments.items():
    if tran_id in complete_segments and tran_id not in residual_tran_ids:
        missing_residuals_by_mix[info["mix_id"]].append(info["tran"])

print(f'\n=== Retry Analysis (MANIFEST={MANIFEST}) ===')
print(f'Total expected segments: {len(expected_segments)}')
print(f'Complete on HF: {len(complete_segments & set(expected_segments.keys()))}')
print(f'Missing segments (need 06b retry): {len(missing_segments)}')
print(f'Missing residuals (need 06c retry): {sum(len(v) for v in missing_residuals_by_mix.values())}')
print(f'Mixes affected: {len(missing_by_mix)}')
print()
for mix_id in sorted(missing_by_mix):
    trans = missing_by_mix[mix_id]
    print(f'  {mix_id}: {len(trans)} missing segments')

if not missing_segments and not any(missing_residuals_by_mix.values()):
    print('\nNothing to retry! All segments and residuals are on HF.')

In [ ]:
# Cell 4 — Retry: re-run Demucs on missing segments + recompute residuals + upload

import sys
if '/kaggle/working/ai-dj/v4' not in sys.path:
    sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import logging
import time
import numpy as np
import librosa
from huggingface_hub import hf_hub_download

logging.basicConfig(level=logging.WARNING)

from aidj.stems.separator import StemSeparator
from aidj.stems.stem_cache import StemCache
from aidj.data.residual import compute_residual, align_track_to_mix_segment
from aidj import config

if not missing_segments and not any(missing_residuals_by_mix.values()):
    print('Nothing to retry!')
else:
    separator = StemSeparator(device="cuda" if torch.cuda.is_available() else "cpu")
    stem_cache = StemCache(DATA_ROOT_PATH)
    print(f'Separator device: {separator.device}')

    def download_mix_audio(mix_id):
        local = DATA_ROOT_PATH / "mixes" / f"{mix_id}.mp3"
        if local.exists():
            return local
        local.parent.mkdir(parents=True, exist_ok=True)
        hf_hub_download(
            repo_id=HF_REPO, filename=f"mixes/{mix_id}.mp3",
            repo_type="dataset", token=HF_TOKEN,
            local_dir=DATA_ROOT,
        )
        return local

    def download_track_stems(tid):
        """Download all 4 stem OGGs for a track from HF."""
        stem_dir = DATA_ROOT_PATH / "stems" / "tracks" / tid
        ext = config.STEM_EXT[config.STEM_FORMAT]
        for stem in config.STEMS:
            local = stem_dir / f"{stem}{ext}"
            if local.exists():
                continue
            stem_dir.mkdir(parents=True, exist_ok=True)
            try:
                hf_hub_download(
                    repo_id=HF_REPO,
                    filename=f"stems/tracks/{tid}/{stem}{ext}",
                    repo_type="dataset", token=HF_TOKEN,
                    local_dir=DATA_ROOT,
                )
            except Exception as e:
                print(f'  Warning: could not download stem {stem} for track {tid}: {e}')

    session_start = time.time()
    total_retried = 0
    total_failed = 0

    # --- Retry missing segments (06b) ---
    all_affected_mixes = sorted(set(list(missing_by_mix.keys()) + list(missing_residuals_by_mix.keys())))

    for mix_idx, mix_id in enumerate(all_affected_mixes):
        print(f'\n{"="*60}')
        missing_segs = missing_by_mix.get(mix_id, [])
        missing_res = missing_residuals_by_mix.get(mix_id, [])
        print(f'[{mix_idx+1}/{len(all_affected_mixes)}] {mix_id}: '
              f'{len(missing_segs)} missing segments, {len(missing_res)} missing residuals')
        print(f'{"="*60}')

        transitions = mixes_with_transitions[mix_id]
        mix_audio = None  # lazy load
        new_seg_ids = []

        # --- 06b retry: separate missing segments (GPU OPTIMIZED) ---
        if missing_segs:
            print(f'\n--- 06b retry: {len(missing_segs)} segments ---')
            mix_path = download_mix_audio(mix_id)
            mix_audio, _ = librosa.load(str(mix_path), sr=config.SR, mono=True)

            for i, tran in enumerate(missing_segs):
                tran_id = tran["tran_id"]

                mix_cue_in = tran.get("mix_cue_in_time_next")
                mix_cue_out = tran.get("mix_cue_out_time_prev")
                mix_start = int(min(mix_cue_in, mix_cue_out) * config.SR)
                mix_end = int(max(mix_cue_in, mix_cue_out) * config.SR)
                segment = mix_audio[mix_start:mix_end]

                try:
                    stem_cache.separate_and_cache_segment(
                        separator, segment, "mix_segments", tran_id
                    )
                    new_seg_ids.append(tran_id)
                    total_retried += 1
                    print(f'  [{i+1}/{len(missing_segs)}] {tran_id}: OK')
                except Exception as e:
                    total_failed += 1
                    print(f'  [{i+1}/{len(missing_segs)}] {tran_id}: FAILED — {e}')
                finally:
                    # GPU OPTIMIZATION: clear cache after EVERY segment, success or failure
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

        # --- 06c retry: recompute residuals for newly separated + previously missing ---
        # Combine: segments we just fixed + segments that had stems but no residual
        need_residuals = set(new_seg_ids) | {t["tran_id"] for t in missing_res}
        residual_trans = [t for t in transitions if t["tran_id"] in need_residuals]

        if residual_trans:
            print(f'\n--- 06c retry: {len(residual_trans)} residuals ---')
            res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
            res_dir.mkdir(parents=True, exist_ok=True)

            # Download track stems needed for residual computation
            track_ids = set()
            for t in residual_trans:
                if t.get("track_id_prev"): track_ids.add(t["track_id_prev"])
                if t.get("track_id_next"): track_ids.add(t["track_id_next"])
            print(f'  Downloading stems for {len(track_ids)} tracks...')
            for tid in sorted(track_ids):
                download_track_stems(tid)

            res_ok = 0
            for tran in residual_trans:
                tran_id = tran["tran_id"]
                prev_id = tran["track_id_prev"]
                next_id = tran["track_id_next"]
                out_path = res_dir / f"{tran_id}.npz"

                mix_cue_in = tran.get("mix_cue_in_time_next")
                mix_cue_out = tran.get("mix_cue_out_time_prev")
                if mix_cue_in is None or mix_cue_out is None:
                    continue
                region_len = int((max(mix_cue_in, mix_cue_out) - min(mix_cue_in, mix_cue_out)) * config.SR)
                if region_len < config.SR:
                    continue

                mix_seg_stems = stem_cache.load_stems("mix_segments", tran_id)
                prev_track_stems = stem_cache.load_stems("tracks", prev_id)
                next_track_stems = stem_cache.load_stems("tracks", next_id)

                if mix_seg_stems is None:
                    print(f'  {tran_id}: still missing segment stems, skip')
                    continue

                residual_data = {}
                if prev_track_stems is not None:
                    t_start = tran.get("track_cue_in_time_prev", 0)
                    aligned = {s: align_track_to_mix_segment(
                        prev_track_stems[s], t_start, region_len, config.SR
                    ) for s in config.STEMS if s in prev_track_stems}
                    for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                        residual_data[f"{stem}_prev"] = spec

                if next_track_stems is not None:
                    t_start = tran.get("track_cue_in_time_next", 0)
                    aligned = {s: align_track_to_mix_segment(
                        next_track_stems[s], t_start, region_len, config.SR
                    ) for s in config.STEMS if s in next_track_stems}
                    for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                        residual_data[f"{stem}_next"] = spec

                if residual_data:
                    np.savez_compressed(str(out_path), **residual_data)
                    res_ok += 1

            print(f'  Residuals computed: {res_ok}/{len(residual_trans)}')

        # --- Upload new stems + residuals ---
        with tempfile.TemporaryDirectory() as staging_dir:
            staging = Path(staging_dir)
            n_files = 0

            # Stage new segment stems
            for tran_id in new_seg_ids:
                seg_dir = stem_cache._stem_dir("mix_segments", tran_id)
                if seg_dir.exists():
                    for f in seg_dir.iterdir():
                        dst = staging / f"stems/mix_segments/{tran_id}/{f.name}"
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(f, dst)
                        n_files += 1

            # Stage new residuals
            res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
            if res_dir.exists():
                for f in res_dir.iterdir():
                    dst = staging / f"results/residuals/{mix_id}/{f.name}"
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, dst)
                    n_files += 1

            if n_files > 0:
                print(f'\n  Uploading {n_files} files for {mix_id}...')
                api.upload_large_folder(
                    folder_path=str(staging),
                    repo_id=HF_REPO,
                    repo_type="dataset",
                )
                print(f'  Upload done')

        # Cleanup local stems to free disk
        for tran_id in new_seg_ids:
            seg_dir = DATA_ROOT_PATH / "stems" / "mix_segments" / tran_id
            if seg_dir.exists():
                shutil.rmtree(seg_dir)
        for tid in track_ids if residual_trans else []:
            stem_dir = DATA_ROOT_PATH / "stems" / "tracks" / tid
            if stem_dir.exists():
                shutil.rmtree(stem_dir)
        mix_mp3 = DATA_ROOT_PATH / "mixes" / f"{mix_id}.mp3"
        if mix_mp3.exists():
            mix_mp3.unlink()

    elapsed = (time.time() - session_start) / 60
    print(f'\n{"="*60}')
    print(f'Retry complete in {elapsed:.1f} min')
    print(f'  Segments retried: {total_retried}')
    print(f'  Still failed: {total_failed}')

In [ ]:
# Cell 5 — Verify: recount missing segments on HF
from collections import defaultdict

print('Re-listing HF files...')
all_hf_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))

seg_stem_files = [f for f in all_hf_files if f.startswith("stems/mix_segments/") and f.endswith(".ogg")]
seg_stems_by_tran = defaultdict(set)
for f in seg_stem_files:
    parts = f.split("/")
    seg_stems_by_tran[parts[2]].add(parts[3].replace(".ogg", ""))

complete_now = {tid for tid, stems in seg_stems_by_tran.items() if stems >= EXPECTED_STEMS}
still_missing = {tid for tid in expected_segments if tid not in complete_now}

residual_files_now = [f for f in all_hf_files if f.startswith("results/residuals/") and f.endswith(".npz")]
residual_tran_ids_now = set(f.split("/")[3].replace(".npz", "") for f in residual_files_now)
missing_res_now = {tid for tid in expected_segments
                   if tid in complete_now and tid not in residual_tran_ids_now}

print(f'\n=== Post-Retry Status ===')
print(f'Expected segments: {len(expected_segments)}')
print(f'Complete segments on HF: {len(complete_now & set(expected_segments.keys()))}')
print(f'Still missing segments: {len(still_missing)}')
print(f'Missing residuals: {len(missing_res_now)}')

if still_missing:
    print(f'\nStill missing segment IDs:')
    for tid in sorted(still_missing)[:30]:
        print(f'  {tid}')
    if len(still_missing) > 30:
        print(f'  ... and {len(still_missing) - 30} more')
else:
    print('\nAll segments and residuals are complete!')